# Análise de Academias

Este notebook cruza população, PIB per capita e número de academias por município.

In [ ]:
# Imports
import pandas as pd
import numpy as np
from unidecode import unidecode
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import statsmodels.api as sm


In [ ]:
# Parâmetros
URL_PIB = 'https://raw.githubusercontent.com/gustavocformoso-collab/opportunity/refs/heads/main/tabela5938.csv'
URL_POP = 'https://raw.githubusercontent.com/gustavocformoso-collab/opportunity/refs/heads/main/tabela6579.csv'
URL_UNIDADES = 'https://raw.githubusercontent.com/gustavocformoso-collab/opportunity/refs/heads/main/unidades.csv'
FILTRAR_REDE = None
BINS_POP = [0, 50000, 200000, 500000, 1_000_000, 10_000_000]
BINS_PIB_PC = [0, 10000, 20000, 40000, 80000, 200000]


In [ ]:
# Funções utilitárias
def normalize(text):
    if pd.isna(text):
        return text
    return unidecode(str(text)).upper().strip()

def detect_columns(df):
    mapping = {}
    for col in df.columns:
        col_norm = normalize(col)
        if 'MUNIC' in col_norm:
            mapping['municipio'] = col
        if col_norm in ('UF', 'SIGLA', 'UNIDADE DA FEDERACAO'):
            mapping['uf'] = col
        if 'VALOR' in col_norm:
            mapping['valor'] = col
        if 'ANO' in col_norm:
            mapping['ano'] = col
        if 'COD' in col_norm and 'MUN' in col_norm:
            mapping['cod_mun'] = col
    return mapping


In [ ]:
# Carregar CSVs do IBGE
pib_raw = pd.read_csv(URL_PIB, sep=';', encoding='latin1')
pop_raw = pd.read_csv(URL_POP, sep=';', encoding='latin1')
print('PIB brutas:', pib_raw.head())
print('Pop brutas:', pop_raw.head())


In [ ]:
# Limpeza PIB
map_pib = detect_columns(pib_raw)
pib = pib_raw.rename(columns=map_pib)
if 'municipio' in pib:
    pib['municipio'] = pib['municipio'].str.replace(r' \(.*\)$', '', regex=True)
    pib['municipio'] = pib['municipio'].apply(normalize)
if 'uf' in pib:
    pib['uf'] = pib['uf'].apply(normalize)
if 'valor' in pib:
    pib['pib'] = pd.to_numeric(pib['valor'], errors='coerce') * 1000
    pib = pib.drop(columns=['valor'])
print(pib.head())


In [ ]:
# Limpeza População
map_pop = detect_columns(pop_raw)
pop = pop_raw.rename(columns=map_pop)
if 'municipio' in pop:
    pop['municipio'] = pop['municipio'].str.replace(r' \(.*\)$', '', regex=True)
    pop['municipio'] = pop['municipio'].apply(normalize)
if 'uf' in pop:
    pop['uf'] = pop['uf'].apply(normalize)
if 'valor' in pop:
    pop['populacao'] = pd.to_numeric(pop['valor'], errors='coerce')
    pop = pop.drop(columns=['valor'])
print(pop.head())


In [ ]:
# Merge PIB e População
merge_cols = ['cod_mun'] if 'cod_mun' in pib.columns and 'cod_mun' in pop.columns else ['municipio', 'uf']
ibge = pd.merge(pop, pib, on=merge_cols, how='inner')
ibge['pib_per_capita'] = ibge['pib'] / ibge['populacao']
print('Após merge IBGE:', ibge.head())


In [ ]:
# Carregar e tratar unidades de academias
unidades = pd.read_csv(URL_UNIDADES)
unidades = unidades.rename(columns={'cidade': 'municipio', 'estado': 'uf'})
unidades['municipio'] = unidades['municipio'].apply(normalize)
unidades['uf'] = unidades['uf'].apply(normalize)
if FILTRAR_REDE:
    unidades = unidades[unidades['rede'].str.upper() == FILTRAR_REDE.upper()]
acad_agg = unidades.groupby(['municipio', 'uf']).size().reset_index(name='n_academias')
print(acad_agg.head())


In [ ]:
# Merge final
final_cols = ['cod_mun'] if 'cod_mun' in ibge.columns else ['municipio', 'uf']
data = pd.merge(ibge, acad_agg, on=final_cols, how='inner')
initial_len = len(data)
data = data.dropna(subset=['populacao', 'pib', 'pib_per_capita'])
data = data[(data['populacao'] > 0) & (data['pib'] > 0) & (data['pib_per_capita'] > 0)]
removed = initial_len - len(data)
print(f'Merge final: {len(data)} linhas, removidas {removed}')
print(data.head())


In [ ]:
# Regressão OLS
X = data[['populacao', 'pib_per_capita']]
X = sm.add_constant(X)
y = data['n_academias']
model = sm.OLS(y, X).fit()
print(model.summary())


In [ ]:
# Scatter 3D
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(data['populacao'], data['pib_per_capita'], data['n_academias'])
ax.set_xlabel('População')
ax.set_ylabel('PIB per capita (R$)')
ax.set_zlabel('Nº de academias')
plt.tight_layout()
plt.savefig('scatter3d.png')
plt.show()


In [ ]:
# Heatmap matriz
pop_bins = pd.cut(data['populacao'], bins=BINS_POP)
pib_bins = pd.cut(data['pib_per_capita'], bins=BINS_PIB_PC)
matriz = data.pivot_table(index=pop_bins, columns=pib_bins, values='n_academias', aggfunc='sum', fill_value=0)
print(matriz)
plt.figure(figsize=(8,6))
plt.imshow(matriz, cmap='viridis', aspect='auto')
plt.colorbar(label='Nº de academias')
plt.xticks(range(len(matriz.columns)), [str(c) for c in matriz.columns], rotation=45)
plt.yticks(range(len(matriz.index)), [str(i) for i in matriz.index])
plt.title('Matriz de Academias por População e PIB per capita')
plt.tight_layout()
plt.savefig('heatmap_matriz.png')
plt.show()


In [ ]:
# Salvar saídas
data[['municipio', 'uf', 'n_academias', 'populacao', 'pib', 'pib_per_capita']].to_csv('base_r3.csv', index=False)
matriz.to_csv('matriz_pop_pib.csv')
print('Arquivos salvos: base_r3.csv, matriz_pop_pib.csv, scatter3d.png, heatmap_matriz.png')
